# Исторический исследовательский код

Эта версия не является основным протоколом статьи. Выводы удалены. Она может требовать исходные локальные файлы и иную схему данных, которые не включены в открытый набор. Для воспроизведения статьи используйте `notebooks/vascular`. Старые метрики не смешиваются с актуальными.


# Анализ госпитальных исходов после балансировки

Ноутбук сопоставляет связи факторов с госпитальной смертью в исходных реальных данных и в наборе после общей балансировки SMOTENC. Анализ выполняется для общей выборки и отдельно для трёх сосудистых бассейнов.

Основными являются результаты реальных пациентов. Результаты сбалансированного набора служат проверкой устойчивости направления и величины эффектов. Значения смертности, p-value и доверительные интервалы после добавления синтетики не интерпретируются как новые клинические доказательства.

## 1. Настройки и загрузка данных

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display, Markdown
from scipy.stats import mannwhitneyu, fisher_exact, spearmanr
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 120
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", lambda value: f"{value:.3f}")

DATA_DIR = Path("data")
REAL_PATH = DATA_DIR / "hospital_outcomes_prepared.csv"
BALANCED_PATH = DATA_DIR / "hospital_outcomes_balanced_final.csv"

real_data = pd.read_csv(REAL_PATH)
balanced_data = pd.read_csv(BALANCED_PATH)

balanced_real = (
    balanced_data.loc[balanced_data["is_synthetic"].eq(0), real_data.columns]
    .reset_index(drop=True)
)

pd.testing.assert_frame_equal(
    real_data.reset_index(drop=True),
    balanced_real,
    check_dtype=False,
    check_exact=False,
)

print("Исходный набор:", real_data.shape)
print("Сбалансированный набор:", balanced_data.shape)
print("Реальные строки полностью сохранены")

## 2. Переменные и группы анализа

Для числовых факторов сравниваются медианы и рангово-бисериальный эффект. Для бинарных факторов рассчитываются отношение шансов и точный критерий Фишера. Поправка Бенджамини — Хохберга применяется отдельно внутри каждого анализа.

Положительный рангово-бисериальный эффект означает, что значения показателя обычно выше у умерших. Отношение шансов больше 1 означает более высокие шансы смерти при наличии бинарного признака.

In [ ]:
continuous_features = [
    "age_years",
    "ef_percent",
    "creatinine_value",
    "total_cholesterol",
    "ldl_mmol_l",
    "glucose_value",
    "crp_value",
    "egfr_value",
    "bmi",
]

binary_features = [
    "female",
    "prior_vascular_surgery",
    "mi_history",
    "stroke_history",
    "ckd_g3plus_proxy",
    "glucose_elevated_proxy",
]

variable_labels = {
    "age_years": "Возраст, лет",
    "ef_percent": "Фракция выброса, %",
    "creatinine_value": "Креатинин",
    "total_cholesterol": "Общий холестерин",
    "ldl_mmol_l": "ЛПНП, ммоль/л",
    "glucose_value": "Глюкоза, ммоль/л",
    "crp_value": "С-реактивный белок",
    "egfr_value": "СКФ",
    "bmi": "Индекс массы тела",
    "female": "Женский пол",
    "prior_vascular_surgery": "Операция на сосудах в анамнезе",
    "mi_history": "Инфаркт миокарда в анамнезе",
    "stroke_history": "ОНМК в анамнезе",
    "ckd_g3plus_proxy": "ХБП G3 и выше",
    "glucose_elevated_proxy": "Глюкоза ≥5,6 ммоль/л",
}

group_labels = {
    "overall": "Общая выборка",
    "peripheral": "Периферические артерии",
    "absh_aortic_aneurysm": "АБШ / аневризма аорты",
    "carotid": "Сонные артерии",
}

scope_order = list(group_labels)


def split_scopes(frame):
    return {
        "overall": frame.copy(),
        "peripheral": frame.loc[frame["vascular_group"].eq("peripheral")].copy(),
        "absh_aortic_aneurysm": frame.loc[
            frame["vascular_group"].eq("absh_aortic_aneurysm")
        ].copy(),
        "carotid": frame.loc[frame["vascular_group"].eq("carotid")].copy(),
    }


real_scopes = split_scopes(real_data)
balanced_scopes = split_scopes(balanced_data)

## 3. Структура исходной и сбалансированной выборок

Смертность и её доверительные интервалы оцениваются только по реальным пациентам. Число смертей в сбалансированном наборе показано как техническая характеристика выполненной генерации.

In [ ]:
def wilson_interval(events, total, z_value=1.959963984540054):
    if total == 0:
        return np.nan, np.nan
    probability = events / total
    denominator = 1 + z_value**2 / total
    center = (
        probability + z_value**2 / (2 * total)
    ) / denominator
    margin = (
        z_value
        * np.sqrt(
            probability * (1 - probability) / total
            + z_value**2 / (4 * total**2)
        )
        / denominator
    )
    return center - margin, center + margin


overview_rows = []

for scope in scope_order:
    real_frame = real_scopes[scope]
    balanced_frame = balanced_scopes[scope]
    real_n = len(real_frame)
    real_deaths = int(real_frame["hospital_death"].sum())
    lower, upper = wilson_interval(real_deaths, real_n)

    overview_rows.append(
        {
            "scope": scope,
            "Группа": group_labels[scope],
            "Реальных пациентов": real_n,
            "Реальных смертей": real_deaths,
            "Наблюдаемая смертность, %": real_deaths / real_n * 100,
            "95% ДИ, нижняя граница, %": lower * 100,
            "95% ДИ, верхняя граница, %": upper * 100,
            "Синтетических смертей": int(
                balanced_frame["is_synthetic"].eq(1).sum()
            ),
            "Смертей после балансировки": int(
                balanced_frame["hospital_death"].sum()
            ),
        }
    )

overview = pd.DataFrame(overview_rows)
display(overview.drop(columns="scope"))

In [ ]:
plot_data = overview.copy()
plot_data["lower_error"] = (
    plot_data["Наблюдаемая смертность, %"]
    - plot_data["95% ДИ, нижняя граница, %"]
)
plot_data["upper_error"] = (
    plot_data["95% ДИ, верхняя граница, %"]
    - plot_data["Наблюдаемая смертность, %"]
)

figure, axis = plt.subplots(figsize=(11, 5.5))
positions = np.arange(len(plot_data))
axis.errorbar(
    positions,
    plot_data["Наблюдаемая смертность, %"],
    yerr=[plot_data["lower_error"], plot_data["upper_error"]],
    fmt="o",
    markersize=8,
    color="#355C7D",
    ecolor="#355C7D",
    capsize=6,
    linewidth=2,
)
axis.set_xticks(positions, plot_data["Группа"], rotation=15, ha="right")
axis.set_ylabel("Наблюдаемая госпитальная смертность, %")
axis.set_xlabel("")
axis.set_title("Реальные пациенты: смертность и 95% доверительные интервалы")
axis.set_ylim(bottom=0)
figure.tight_layout()
plt.show()
plt.close(figure)

### 3.1. Попарное сравнение смертности сосудистых бассейнов

Сравнение выполняется только по реальным пациентам. Используются точный критерий Фишера и отношение шансов с 95% доверительным интервалом.

In [ ]:
vascular_pairs = [
    ("peripheral", "absh_aortic_aneurysm"),
    ("peripheral", "carotid"),
    ("absh_aortic_aneurysm", "carotid"),
]

pairwise_rows = []

for first_scope, second_scope in vascular_pairs:
    first = real_scopes[first_scope]
    second = real_scopes[second_scope]
    a = int(first["hospital_death"].sum())
    b = int(len(first) - a)
    c = int(second["hospital_death"].sum())
    d = int(len(second) - c)
    values = np.array([a, b, c, d], dtype=float)
    if (values == 0).any():
        values += 0.5
    a_value, b_value, c_value, d_value = values
    odds_ratio = a_value * d_value / (b_value * c_value)
    standard_error = np.sqrt(
        1 / a_value + 1 / b_value + 1 / c_value + 1 / d_value
    )
    lower = np.exp(np.log(odds_ratio) - 1.959963984540054 * standard_error)
    upper = np.exp(np.log(odds_ratio) + 1.959963984540054 * standard_error)
    p_value = float(
        fisher_exact([[a, b], [c, d]], alternative="two-sided").pvalue
    )

    pairwise_rows.append(
        {
            "Первая группа": group_labels[first_scope],
            "Смертей в первой": a,
            "Пациентов в первой": a + b,
            "Вторая группа": group_labels[second_scope],
            "Смертей во второй": c,
            "Пациентов во второй": c + d,
            "OR первой против второй": odds_ratio,
            "95% ДИ, нижняя граница": lower,
            "95% ДИ, верхняя граница": upper,
            "p_value": p_value,
        }
    )

pairwise_mortality = pd.DataFrame(pairwise_rows)
pairwise_mortality["p_fdr"] = multipletests(
    pairwise_mortality["p_value"],
    method="fdr_bh",
)[1]
display(pairwise_mortality)

## 4. Описательный анализ по сосудистым бассейнам

Для каждого бассейна сопоставляются все реальные пациенты, реальные выжившие, реальные умершие и умершие после балансировки. Группа «умершие после балансировки» содержит реальные и синтетические летальные исходы. Выжившие после балансировки не выводятся повторно, поскольку это те же реальные пациенты.

In [ ]:
vascular_scopes = [
    "peripheral",
    "absh_aortic_aneurysm",
    "carotid",
]


def median_iqr_text(series):
    observed = series.dropna()
    if observed.empty:
        return "нет данных"
    return (
        f"{observed.median():.2f} "
        f"[{observed.quantile(0.25):.2f}; {observed.quantile(0.75):.2f}]"
    )


def prevalence_text(series):
    observed = series.dropna()
    if observed.empty:
        return "нет данных"
    events = int(observed.eq(1).sum())
    return f"{events}/{len(observed)} ({observed.eq(1).mean() * 100:.1f}%)"


continuous_descriptive_rows = []
binary_descriptive_rows = []

for scope in vascular_scopes:
    real_frame = real_scopes[scope]
    balanced_frame = balanced_scopes[scope]
    series_frames = {
        "Все реальные": real_frame,
        "Живы, реальные": real_frame.loc[real_frame["hospital_death"].eq(0)],
        "Умерли, реальные": real_frame.loc[real_frame["hospital_death"].eq(1)],
        "Умерли после балансировки": balanced_frame.loc[
            balanced_frame["hospital_death"].eq(1)
        ],
    }

    for variable in continuous_features:
        row = {
            "scope": scope,
            "Группа": group_labels[scope],
            "Показатель": variable_labels[variable],
        }
        for series_label, frame in series_frames.items():
            row[series_label] = median_iqr_text(frame[variable])
        continuous_descriptive_rows.append(row)

    for variable in binary_features:
        row = {
            "scope": scope,
            "Группа": group_labels[scope],
            "Показатель": variable_labels[variable],
        }
        for series_label, frame in series_frames.items():
            row[series_label] = prevalence_text(frame[variable])
        binary_descriptive_rows.append(row)

continuous_descriptive = pd.DataFrame(continuous_descriptive_rows)
binary_descriptive = pd.DataFrame(binary_descriptive_rows)

for scope in vascular_scopes:
    display(Markdown(f"### {group_labels[scope]}"))
    display(Markdown("**Числовые показатели: медиана [Q1; Q3]**"))
    display(
        continuous_descriptive.loc[
            continuous_descriptive["scope"].eq(scope)
        ].drop(columns=["scope", "Группа"])
    )
    display(Markdown("**Бинарные показатели: n/N (%)**"))
    display(
        binary_descriptive.loc[
            binary_descriptive["scope"].eq(scope)
        ].drop(columns=["scope", "Группа"])
    )

### 4.1. Распределения числовых показателей

Графики показывают реальные значения выживших, реальные значения умерших и распределение умерших после добавления синтетических наблюдений.

In [ ]:
distribution_palette = {
    "Живы, реальные": "#355C7D",
    "Умерли, реальные": "#8F5B70",
    "Умерли после балансировки": "#C06C84",
}

for scope in vascular_scopes:
    real_frame = real_scopes[scope]
    balanced_frame = balanced_scopes[scope]
    plot_frame = pd.concat(
        [
            real_frame.loc[real_frame["hospital_death"].eq(0)].assign(
                series="Живы, реальные"
            ),
            real_frame.loc[real_frame["hospital_death"].eq(1)].assign(
                series="Умерли, реальные"
            ),
            balanced_frame.loc[balanced_frame["hospital_death"].eq(1)].assign(
                series="Умерли после балансировки"
            ),
        ],
        ignore_index=True,
    )

    figure, axes = plt.subplots(3, 3, figsize=(18, 15))

    for axis, variable in zip(axes.flat, continuous_features):
        sns.boxplot(
            data=plot_frame,
            x="series",
            y=variable,
            order=list(distribution_palette),
            hue="series",
            palette=distribution_palette,
            legend=False,
            showfliers=True,
            ax=axis,
        )
        axis.set_title(variable_labels[variable])
        axis.set_xlabel("")
        axis.set_ylabel("")
        axis.tick_params(axis="x", rotation=18)

    figure.suptitle(
        f"Числовые показатели: {group_labels[scope]}",
        y=1.01,
    )
    figure.tight_layout()
    plt.show()
    plt.close(figure)

### 4.2. Распространённость бинарных признаков

In [ ]:
binary_plot_rows = []

for scope in vascular_scopes:
    real_frame = real_scopes[scope]
    balanced_frame = balanced_scopes[scope]
    source_frames = {
        "Живы, реальные": real_frame.loc[real_frame["hospital_death"].eq(0)],
        "Умерли, реальные": real_frame.loc[real_frame["hospital_death"].eq(1)],
        "Умерли после балансировки": balanced_frame.loc[
            balanced_frame["hospital_death"].eq(1)
        ],
    }

    for source, frame in source_frames.items():
        for variable in binary_features:
            observed = frame[variable].dropna()
            binary_plot_rows.append(
                {
                    "scope": scope,
                    "Группа": group_labels[scope],
                    "Серия": source,
                    "Показатель": variable_labels[variable],
                    "Распространённость, %": (
                        observed.eq(1).mean() * 100
                        if not observed.empty
                        else np.nan
                    ),
                }
            )

binary_plot_data = pd.DataFrame(binary_plot_rows)
figure, axes = plt.subplots(1, 3, figsize=(22, 8), sharex=True)

for axis, scope in zip(axes, vascular_scopes):
    sns.barplot(
        data=binary_plot_data.loc[binary_plot_data["scope"].eq(scope)],
        x="Распространённость, %",
        y="Показатель",
        hue="Серия",
        hue_order=list(distribution_palette),
        palette=distribution_palette,
        ax=axis,
    )
    axis.set_xlim(0, 100)
    axis.set_title(group_labels[scope])
    axis.set_xlabel("Распространённость среди доступных наблюдений, %")
    axis.set_ylabel("")
    if axis is not axes[0]:
        axis.get_legend().remove()

handles, labels = axes[0].get_legend_handles_labels()
axes[0].get_legend().remove()
figure.suptitle("Бинарные признаки по сосудистым бассейнам", y=0.98)
figure.legend(
    handles,
    labels,
    loc="upper center",
    bbox_to_anchor=(0.5, 0.94),
    ncol=3,
    frameon=False,
)
figure.tight_layout(rect=[0, 0, 1, 0.88])
plt.show()
plt.close(figure)

### 4.3. Госпитальная смертность по полу

Показатели рассчитываются только на реальных пациентах. Доверительные интервалы определены методом Уилсона.

In [ ]:
sex_rows = []

for scope in vascular_scopes:
    frame = real_scopes[scope]
    for female_value, sex_label in [(0, "Мужчины"), (1, "Женщины")]:
        subset = frame.loc[frame["female"].eq(female_value)]
        patients = len(subset)
        deaths = int(subset["hospital_death"].sum())
        lower, upper = wilson_interval(deaths, patients)
        sex_rows.append(
            {
                "scope": scope,
                "Группа": group_labels[scope],
                "Пол": sex_label,
                "Пациентов": patients,
                "Смертей": deaths,
                "Смертность, %": deaths / patients * 100 if patients else np.nan,
                "95% ДИ, нижняя граница, %": lower * 100,
                "95% ДИ, верхняя граница, %": upper * 100,
            }
        )

sex_summary = pd.DataFrame(sex_rows)
display(sex_summary.drop(columns="scope"))

figure, axes = plt.subplots(1, 3, figsize=(17, 5), sharey=True)

for axis, scope in zip(axes, vascular_scopes):
    plot_frame = sex_summary.loc[sex_summary["scope"].eq(scope)].reset_index(drop=True)
    positions = np.arange(len(plot_frame))
    lower_error = plot_frame["Смертность, %"] - plot_frame["95% ДИ, нижняя граница, %"]
    upper_error = plot_frame["95% ДИ, верхняя граница, %"] - plot_frame["Смертность, %"]
    axis.errorbar(
        positions,
        plot_frame["Смертность, %"],
        yerr=[lower_error, upper_error],
        fmt="o",
        markersize=8,
        color="#355C7D",
        capsize=5,
        linewidth=2,
    )
    axis.set_xticks(positions, plot_frame["Пол"])
    axis.set_title(group_labels[scope])
    axis.set_xlabel("")
    axis.set_ylabel("Наблюдаемая смертность, %" if axis is axes[0] else "")

upper_limit = sex_summary["95% ДИ, верхняя граница, %"].max() * 1.08
axes[0].set_ylim(0, upper_limit)

figure.suptitle("Реальные пациенты: смертность по полу", y=0.98)
figure.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()
plt.close(figure)

## 5. Статистические функции

In [ ]:
def add_fdr(table):
    result = table.copy()
    result["p_fdr"] = np.nan
    valid = result["p_value"].notna()
    if valid.any():
        result.loc[valid, "p_fdr"] = multipletests(
            result.loc[valid, "p_value"],
            method="fdr_bh",
        )[1]
    return result


def continuous_analysis(frame):
    rows = []

    for variable in continuous_features:
        alive = frame.loc[frame["hospital_death"].eq(0), variable].dropna()
        deceased = frame.loc[frame["hospital_death"].eq(1), variable].dropna()

        if not alive.empty and not deceased.empty:
            test = mannwhitneyu(deceased, alive, alternative="two-sided")
            effect = 2 * test.statistic / (len(deceased) * len(alive)) - 1
            p_value = float(test.pvalue)
        else:
            effect = np.nan
            p_value = np.nan

        rows.append(
            {
                "variable": variable,
                "Показатель": variable_labels[variable],
                "Живы, n": len(alive),
                "Живы, медиана": alive.median() if not alive.empty else np.nan,
                "Живы, Q1": alive.quantile(0.25) if not alive.empty else np.nan,
                "Живы, Q3": alive.quantile(0.75) if not alive.empty else np.nan,
                "Умерли, n": len(deceased),
                "Умерли, медиана": deceased.median() if not deceased.empty else np.nan,
                "Умерли, Q1": deceased.quantile(0.25) if not deceased.empty else np.nan,
                "Умерли, Q3": deceased.quantile(0.75) if not deceased.empty else np.nan,
                "rank_biserial": effect,
                "p_value": p_value,
            }
        )

    return add_fdr(pd.DataFrame(rows))


def odds_ratio_statistics(a, b, c, d):
    values = np.array([a, b, c, d], dtype=float)
    if (values == 0).any():
        values += 0.5
    a_value, b_value, c_value, d_value = values
    odds_ratio = (a_value * d_value) / (b_value * c_value)
    standard_error = np.sqrt(
        1 / a_value + 1 / b_value + 1 / c_value + 1 / d_value
    )
    lower = np.exp(np.log(odds_ratio) - 1.959963984540054 * standard_error)
    upper = np.exp(np.log(odds_ratio) + 1.959963984540054 * standard_error)
    return odds_ratio, lower, upper, np.log(odds_ratio)


def binary_analysis(frame):
    rows = []

    for variable in binary_features:
        observed = frame[[variable, "hospital_death"]].dropna()
        a = int(((observed[variable] == 1) & (observed["hospital_death"] == 1)).sum())
        b = int(((observed[variable] == 1) & (observed["hospital_death"] == 0)).sum())
        c = int(((observed[variable] == 0) & (observed["hospital_death"] == 1)).sum())
        d = int(((observed[variable] == 0) & (observed["hospital_death"] == 0)).sum())

        has_two_exposures = (a + b) > 0 and (c + d) > 0
        has_two_outcomes = (a + c) > 0 and (b + d) > 0

        if has_two_exposures and has_two_outcomes:
            odds_ratio, lower, upper, log_or = odds_ratio_statistics(a, b, c, d)
            p_value = float(
                fisher_exact([[a, b], [c, d]], alternative="two-sided").pvalue
            )
        else:
            odds_ratio = lower = upper = log_or = p_value = np.nan

        rows.append(
            {
                "variable": variable,
                "Показатель": variable_labels[variable],
                "Доступно, n": len(observed),
                "Признак есть, n": a + b,
                "Смертей при наличии": a,
                "Смертность при наличии, %": a / (a + b) * 100 if (a + b) else np.nan,
                "Признак отсутствует, n": c + d,
                "Смертей при отсутствии": c,
                "Смертность при отсутствии, %": c / (c + d) * 100 if (c + d) else np.nan,
                "odds_ratio": odds_ratio,
                "or_ci_lower": lower,
                "or_ci_upper": upper,
                "log_or": log_or,
                "p_value": p_value,
            }
        )

    return add_fdr(pd.DataFrame(rows))

## 6. Основной анализ реальных пациентов

Таблицы ниже являются основной статистической частью исследования. Для группы сонных артерий результаты отражают только один реальный летальный исход и поэтому рассматриваются как описательные.

In [ ]:
real_continuous = {}
real_binary = {}
balanced_continuous = {}
balanced_binary = {}

for scope in scope_order:
    real_continuous[scope] = continuous_analysis(real_scopes[scope])
    real_binary[scope] = binary_analysis(real_scopes[scope])
    balanced_continuous[scope] = continuous_analysis(balanced_scopes[scope])
    balanced_binary[scope] = binary_analysis(balanced_scopes[scope])

for scope in scope_order:
    display(Markdown(f"### {group_labels[scope]}"))
    display(Markdown("**Числовые факторы**"))
    display(
        real_continuous[scope][
            [
                "Показатель",
                "Живы, n",
                "Живы, медиана",
                "Умерли, n",
                "Умерли, медиана",
                "rank_biserial",
                "p_value",
                "p_fdr",
            ]
        ].sort_values("p_fdr", na_position="last")
    )
    display(Markdown("**Бинарные факторы**"))
    display(
        real_binary[scope][
            [
                "Показатель",
                "Доступно, n",
                "Смертей при наличии",
                "Смертей при отсутствии",
                "odds_ratio",
                "or_ci_lower",
                "or_ci_upper",
                "p_value",
                "p_fdr",
            ]
        ].sort_values("p_fdr", na_position="last")
    )

## 7. Сопоставление эффектов до и после балансировки

Сравнивается не количество статистически значимых p-value, а устойчивость величины и направления эффекта. Смена знака означает, что связь после добавления синтетики изменила направление. Для очень малых эффектов такая смена может быть следствием колебаний около нуля.

In [ ]:
def effect_direction(value, neutral_limit):
    if pd.isna(value):
        return "нет оценки"
    if value > neutral_limit:
        return "положительное"
    if value < -neutral_limit:
        return "отрицательное"
    return "около нуля"


continuous_stability_rows = []
binary_stability_rows = []

for scope in scope_order:
    continuous_merged = real_continuous[scope].merge(
        balanced_continuous[scope],
        on=["variable", "Показатель"],
        suffixes=("_real", "_balanced"),
    )

    for _, row in continuous_merged.iterrows():
        direction_real = effect_direction(row["rank_biserial_real"], 0.05)
        direction_balanced = effect_direction(row["rank_biserial_balanced"], 0.05)
        continuous_stability_rows.append(
            {
                "scope": scope,
                "Группа": group_labels[scope],
                "variable": row["variable"],
                "Показатель": row["Показатель"],
                "Эффект, реальные": row["rank_biserial_real"],
                "Эффект, сбалансированные": row["rank_biserial_balanced"],
                "Абсолютное изменение": abs(
                    row["rank_biserial_balanced"] - row["rank_biserial_real"]
                ),
                "Направление, реальные": direction_real,
                "Направление, сбалансированные": direction_balanced,
                "Сравнение доступно": pd.notna(row["rank_biserial_real"])
                and pd.notna(row["rank_biserial_balanced"]),
                "Направление сохранено": pd.notna(row["rank_biserial_real"])
                and pd.notna(row["rank_biserial_balanced"])
                and direction_real == direction_balanced,
                "p, реальные": row["p_value_real"],
                "p FDR, реальные": row["p_fdr_real"],
                "p, после балансировки, исследовательское": row["p_value_balanced"],
            }
        )

    binary_merged = real_binary[scope].merge(
        balanced_binary[scope],
        on=["variable", "Показатель"],
        suffixes=("_real", "_balanced"),
    )

    for _, row in binary_merged.iterrows():
        direction_real = effect_direction(row["log_or_real"], 0.10)
        direction_balanced = effect_direction(row["log_or_balanced"], 0.10)
        binary_stability_rows.append(
            {
                "scope": scope,
                "Группа": group_labels[scope],
                "variable": row["variable"],
                "Показатель": row["Показатель"],
                "OR, реальные": row["odds_ratio_real"],
                "OR, сбалансированные": row["odds_ratio_balanced"],
                "log(OR), реальные": row["log_or_real"],
                "log(OR), сбалансированные": row["log_or_balanced"],
                "Абсолютное изменение log(OR)": abs(
                    row["log_or_balanced"] - row["log_or_real"]
                ),
                "Направление, реальные": direction_real,
                "Направление, сбалансированные": direction_balanced,
                "Сравнение доступно": pd.notna(row["log_or_real"])
                and pd.notna(row["log_or_balanced"]),
                "Направление сохранено": pd.notna(row["log_or_real"])
                and pd.notna(row["log_or_balanced"])
                and direction_real == direction_balanced,
                "p, реальные": row["p_value_real"],
                "p FDR, реальные": row["p_fdr_real"],
                "p, после балансировки, исследовательское": row["p_value_balanced"],
            }
        )

continuous_stability = pd.DataFrame(continuous_stability_rows)
binary_stability = pd.DataFrame(binary_stability_rows)

for scope in scope_order:
    display(Markdown(f"### {group_labels[scope]}"))
    display(Markdown("**Устойчивость числовых факторов**"))
    display(
        continuous_stability.loc[
            continuous_stability["scope"].eq(scope),
            [
                "Показатель",
                "Эффект, реальные",
                "Эффект, сбалансированные",
                "Абсолютное изменение",
                "Сравнение доступно",
                "Направление сохранено",
                "p FDR, реальные",
                "p, после балансировки, исследовательское",
            ],
        ].sort_values("Абсолютное изменение", ascending=False)
    )
    display(Markdown("**Устойчивость бинарных факторов**"))
    display(
        binary_stability.loc[
            binary_stability["scope"].eq(scope),
            [
                "Показатель",
                "OR, реальные",
                "OR, сбалансированные",
                "Абсолютное изменение log(OR)",
                "Сравнение доступно",
                "Направление сохранено",
                "p FDR, реальные",
                "p, после балансировки, исследовательское",
            ],
        ].sort_values("Абсолютное изменение log(OR)", ascending=False)
    )

## 8. Графики устойчивости эффектов

Линия соединяет оценку на реальных данных с оценкой после балансировки. Чем короче линия, тем меньше балансировка изменила величину эффекта. Пересечение нулевой линии для числовых факторов или линии OR = 1 для бинарных указывает на смену направления связи.

In [ ]:
figure, axes = plt.subplots(2, 2, figsize=(17, 12))

for axis, scope in zip(axes.flat, scope_order):
    plot_frame = (
        continuous_stability.loc[continuous_stability["scope"].eq(scope)]
        .dropna(subset=["Эффект, реальные", "Эффект, сбалансированные"])
        .sort_values("Эффект, реальные")
        .reset_index(drop=True)
    )
    y_positions = np.arange(len(plot_frame))

    for position, row in plot_frame.iterrows():
        axis.plot(
            [row["Эффект, реальные"], row["Эффект, сбалансированные"]],
            [position, position],
            color="#9E9E9E",
            linewidth=2,
            zorder=1,
        )

    axis.scatter(
        plot_frame["Эффект, реальные"],
        y_positions,
        color="#355C7D",
        s=55,
        label="Реальные",
        zorder=2,
    )
    axis.scatter(
        plot_frame["Эффект, сбалансированные"],
        y_positions,
        color="#C06C84",
        s=55,
        label="После балансировки",
        zorder=2,
    )
    axis.axvline(0, color="black", linewidth=1, linestyle="--")
    axis.set_yticks(y_positions, plot_frame["Показатель"])
    axis.set_xlim(-1.05, 1.05)
    axis.set_xlabel("Рангово-бисериальный эффект")
    axis.set_ylabel("")
    axis.set_title(group_labels[scope])

handles, labels = axes.flat[0].get_legend_handles_labels()
figure.legend(handles, labels, loc="upper center", ncol=2, frameon=False)
figure.suptitle("Числовые факторы: реальные и сбалансированные данные", y=0.98)
figure.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()
plt.close(figure)

In [ ]:
figure, axes = plt.subplots(2, 2, figsize=(17, 10))

for axis, scope in zip(axes.flat, scope_order):
    plot_frame = (
        binary_stability.loc[binary_stability["scope"].eq(scope)]
        .dropna(subset=["OR, реальные", "OR, сбалансированные"])
        .sort_values("OR, реальные")
        .reset_index(drop=True)
    )
    y_positions = np.arange(len(plot_frame))

    for position, row in plot_frame.iterrows():
        axis.plot(
            [row["OR, реальные"], row["OR, сбалансированные"]],
            [position, position],
            color="#9E9E9E",
            linewidth=2,
            zorder=1,
        )

    axis.scatter(
        plot_frame["OR, реальные"],
        y_positions,
        color="#355C7D",
        s=55,
        label="Реальные",
        zorder=2,
    )
    axis.scatter(
        plot_frame["OR, сбалансированные"],
        y_positions,
        color="#C06C84",
        s=55,
        label="После балансировки",
        zorder=2,
    )
    axis.axvline(1, color="black", linewidth=1, linestyle="--")
    axis.set_xscale("log")
    axis.set_yticks(y_positions, plot_frame["Показатель"])
    axis.set_xlabel("Отношение шансов, логарифмическая шкала")
    axis.set_ylabel("")
    axis.set_title(group_labels[scope])

handles, labels = axes.flat[0].get_legend_handles_labels()
figure.legend(handles, labels, loc="upper center", ncol=2, frameon=False)
figure.suptitle("Бинарные факторы: реальные и сбалансированные данные", y=0.98)
figure.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()
plt.close(figure)

## 9. Сводка устойчивости

Сводка показывает число факторов, для которых направление эффекта сохранилось. Это технический критерий согласованности, а не доказательство причинной связи.

In [ ]:
stability_summary_rows = []

for scope in scope_order:
    continuous_scope = continuous_stability.loc[
        continuous_stability["scope"].eq(scope)
    ]
    binary_scope = binary_stability.loc[
        binary_stability["scope"].eq(scope)
    ]

    stability_summary_rows.append(
        {
            "Группа": group_labels[scope],
            "Числовых направлений сохранено": int(
                continuous_scope["Направление сохранено"].sum()
            ),
            "Числовых сравнений доступно": int(
                continuous_scope["Сравнение доступно"].sum()
            ),
            "Медианное изменение числового эффекта": continuous_scope[
                "Абсолютное изменение"
            ].median(),
            "Бинарных направлений сохранено": int(
                binary_scope["Направление сохранено"].sum()
            ),
            "Бинарных сравнений доступно": int(
                binary_scope["Сравнение доступно"].sum()
            ),
            "Медианное изменение log(OR)": binary_scope[
                "Абсолютное изменение log(OR)"
            ].median(),
        }
    )

stability_summary = pd.DataFrame(stability_summary_rows)
display(stability_summary)

### 9.1. Согласованность ранжирования эффектов

Корреляция Спирмена показывает, насколько одинаково факторы упорядочиваются по величине эффекта до и после балансировки. Значение ближе к 1 означает более устойчивое ранжирование.

In [ ]:
concordance_rows = []

for scope in scope_order:
    continuous_scope = continuous_stability.loc[
        continuous_stability["scope"].eq(scope)
        & continuous_stability["Сравнение доступно"]
    ]
    binary_scope = binary_stability.loc[
        binary_stability["scope"].eq(scope)
        & binary_stability["Сравнение доступно"]
    ]

    for factor_type, frame, real_column, balanced_column, difference_column in [
        (
            "Числовые",
            continuous_scope,
            "Эффект, реальные",
            "Эффект, сбалансированные",
            "Абсолютное изменение",
        ),
        (
            "Бинарные",
            binary_scope,
            "log(OR), реальные",
            "log(OR), сбалансированные",
            "Абсолютное изменение log(OR)",
        ),
    ]:
        if len(frame) >= 3:
            rank_correlation = float(
                spearmanr(frame[real_column], frame[balanced_column]).statistic
            )
        else:
            rank_correlation = np.nan

        concordance_rows.append(
            {
                "Группа": group_labels[scope],
                "Тип факторов": factor_type,
                "Сравнено факторов": len(frame),
                "Корреляция рангов эффектов": rank_correlation,
                "Медианное абсолютное изменение": frame[difference_column].median(),
                "Направлений сохранено, %": (
                    frame["Направление сохранено"].mean() * 100
                    if not frame.empty
                    else np.nan
                ),
            }
        )

effect_concordance = pd.DataFrame(concordance_rows)
display(effect_concordance)

In [ ]:
for scope in scope_order:
    display(Markdown(f"### {group_labels[scope]}: наибольшие эффекты в реальных данных"))

    continuous_focus = (
        continuous_stability.loc[continuous_stability["scope"].eq(scope)]
        .assign(abs_real=lambda frame: frame["Эффект, реальные"].abs())
        .sort_values("abs_real", ascending=False)
        [[
            "Показатель",
            "Эффект, реальные",
            "Эффект, сбалансированные",
            "Направление сохранено",
            "p FDR, реальные",
        ]]
        .head(5)
    )

    binary_focus = (
        binary_stability.loc[binary_stability["scope"].eq(scope)]
        .assign(abs_real=lambda frame: frame["log(OR), реальные"].abs())
        .sort_values("abs_real", ascending=False)
        [[
            "Показатель",
            "OR, реальные",
            "OR, сбалансированные",
            "Направление сохранено",
            "p FDR, реальные",
        ]]
        .head(5)
    )

    display(Markdown("**Числовые факторы**"))
    display(continuous_focus)
    display(Markdown("**Бинарные факторы**"))
    display(binary_focus)

## 10. Категории фракции выброса

Категории ФВ выводятся описательно. Из-за малого числа смертей в отдельных бассейнах формальное сравнение многокатегориального признака может быть нестабильным.

In [ ]:
ef_rows = []

for data_label, scopes in [
    ("Реальные", real_scopes),
    ("После балансировки", balanced_scopes),
]:
    for scope in scope_order:
        frame = scopes[scope].dropna(subset=["ef_category"])
        counts = (
            frame.groupby(
                ["hospital_death", "ef_category"],
                observed=True,
            )
            .size()
            .rename("n")
            .reset_index()
        )
        if not counts.empty:
            counts["Процент внутри исхода"] = (
                counts.groupby("hospital_death")["n"]
                .transform(lambda values: values / values.sum() * 100)
            )
        counts["Данные"] = data_label
        counts["Группа"] = group_labels[scope]
        ef_rows.append(counts)

ef_summary = pd.concat(ef_rows, ignore_index=True)
ef_summary["Исход"] = ef_summary["hospital_death"].map({0: "Жив", 1: "Умер"})

for scope in scope_order:
    display(Markdown(f"### {group_labels[scope]}"))
    display(
        ef_summary.loc[
            ef_summary["Группа"].eq(group_labels[scope]),
            ["Данные", "Исход", "ef_category", "n", "Процент внутри исхода"],
        ]
    )

## 11. Краткий итог основного анализа

В таблицу включены сигналы с `p FDR ≤ 0,05` на реальных данных. Они остаются исследовательскими ассоциациями и не обозначают причинное влияние.

In [ ]:
primary_signal_rows = []

for scope in scope_order:
    continuous_signals = continuous_stability.loc[
        continuous_stability["scope"].eq(scope)
        & continuous_stability["p FDR, реальные"].le(0.05)
    ]
    binary_signals = binary_stability.loc[
        binary_stability["scope"].eq(scope)
        & binary_stability["p FDR, реальные"].le(0.05)
    ]

    for _, row in continuous_signals.iterrows():
        primary_signal_rows.append(
            {
                "Группа": group_labels[scope],
                "Тип": "Числовой",
                "Показатель": row["Показатель"],
                "Эффект на реальных данных": row["Эффект, реальные"],
                "p FDR": row["p FDR, реальные"],
                "Направление после балансировки сохранено": row[
                    "Направление сохранено"
                ],
            }
        )

    for _, row in binary_signals.iterrows():
        primary_signal_rows.append(
            {
                "Группа": group_labels[scope],
                "Тип": "Бинарный",
                "Показатель": row["Показатель"],
                "Эффект на реальных данных": row["OR, реальные"],
                "p FDR": row["p FDR, реальные"],
                "Направление после балансировки сохранено": row[
                    "Направление сохранено"
                ],
            }
        )

primary_signals = pd.DataFrame(primary_signal_rows)

if primary_signals.empty:
    display(Markdown("После поправки FDR сигналов с p ≤ 0,05 не выявлено."))
else:
    display(primary_signals.sort_values(["Группа", "p FDR"]))

## 12. Технический контроль пропусков

Информация о пропусках вынесена в конец и не используется как самостоятельный клинический результат.

In [ ]:
missingness_rows = []

for scope in scope_order:
    for variable in continuous_features:
        real_missing = real_scopes[scope][variable].isna().mean() * 100
        balanced_missing = balanced_scopes[scope][variable].isna().mean() * 100
        missingness_rows.append(
            {
                "Группа": group_labels[scope],
                "Показатель": variable_labels[variable],
                "Пропуски, реальные, %": real_missing,
                "Пропуски, после балансировки, %": balanced_missing,
                "Абсолютное изменение, п.п.": abs(balanced_missing - real_missing),
            }
        )

missingness_summary = pd.DataFrame(missingness_rows)

for scope in scope_order:
    display(Markdown(f"### {group_labels[scope]}"))
    display(
        missingness_summary.loc[
            missingness_summary["Группа"].eq(group_labels[scope])
        ].sort_values("Пропуски, реальные, %", ascending=False)
    )

## 13. Правила интерпретации

- Наблюдаемая смертность и её доверительный интервал относятся только к реальным пациентам.
- Основными статистическими результатами являются эффекты и p-value исходного датасета.
- Сбалансированный набор используется для проверки сохранения направления и порядка величин эффектов.
- Уменьшение p-value после балансировки не является подтверждением фактора риска.
- Результаты для периферического бассейна основаны на пяти реальных смертях, для сонных артерий — на одной; они требуют особенно осторожной интерпретации.
- Синтетические строки не считаются новыми независимыми клиническими событиями.